# Preprocessing


## Notebook Preferences


In [ ]:
%load_ext autoreload
%autoreload 2

## Importing Libraries


In [ ]:
import lamindb as ln
import nbl
import spatialdata as sd
import scanpy as sc
import annsel as an
import squidpy as sq
from multispaeti import MultispatiPCA
import spatialleiden as sl
from sklearn_ann.kneighbors.annoy import AnnoyTransformer

In [ ]:
ln.settings.set_git_repo = "https://github.com/karadavis-lab/nbl.git"
ln.track("YOQHjxZYuPsW0000")

In [ ]:
ln.setup.settings.instance

In [ ]:
nbl_sdata: sd.SpatialData = sd.read_zarr(ln.Artifact.get(description="NBL Tissue Samples", is_latest=True).path)

In [ ]:
sdata = nbl.pp.arcsinh_transform(
    sdata=nbl_sdata,
    table_names={"whole_cell": "wc_arcsinh", "nuclear": "nu_arcsinh"},
    shift_factor=0,
    scale_factor=150,
    write=False,
    inplace=False,
)

NBL-5-R9C7, NBL-31-R1C7, NBL-17-R11C10 have 5 or less NBL cells therefore per-fov graph neighbors cannot be computed.


In [ ]:
nbl_adata = sdata.tables["wc_arcsinh"].an.filter(
    an.var_names().is_in(nbl.ln.Neuroblastoma_Markers + nbl.ln.Neuroblastoma_Extra_Markers),
    an.obs_col("pixie_cluster") == "NBL_cell",
    ~an.obs_col("fov").is_in(["NBL-5-R9C7", "NBL-31-R1C7", "NBL-17-R11C10"]),
)

In [ ]:
sc.pp.neighbors(
    nbl_adata,
    n_neighbors=15,
    key_added="sc_neighbors",
    transformer=AnnoyTransformer(n_neighbors=15, n_trees=100),
)

In [ ]:
msPCA = MultispatiPCA(n_components=5, connectivity=nbl_adata.obsp["sc_neighbors_connectivities"])

In [ ]:
X_transformed = msPCA.fit_transform(nbl_adata.X)

In [ ]:
nbl_adata.obsm["msPCA"] = X_transformed

In [ ]:
sc.pp.neighbors(
    nbl_adata,
    n_neighbors=15,
    key_added="msPCA_neighbors",
    transformer=AnnoyTransformer(n_neighbors=15, n_trees=100),
    use_rep="msPCA",
)

In [ ]:
sq.gr.spatial_neighbors(
    nbl_adata, spatial_key="spatial", library_key="region", coord_type="generic", key_added="sq_spatial", n_neighs=15
)

In [ ]:
nbl_adata

In [ ]:
nbl_adata.obsp["sl_connectivities"] = sl.distance2connectivity(nbl_adata.obsp["sq_spatial_distances"])

In [ ]:
# nbl_fov = nbl_adata.an.filter(an.obs_col("fov") == "NBL-1-R5C8")

sl.spatialleiden(
    nbl_adata,
    layer_ratio=1.8,
    directed=(True, False),
    latent_distance_key="sc_neighbors_connectivities",
    spatial_distance_key="msPCA_neighbors_connectivities",
    key_added="spatialleiden",
)

In [ ]:
print(nbl_adata.obs["spatialleiden"].value_counts())
print(nbl_adata.obs["fov"].value_counts())

In [ ]:
nbl_sdata.tables["nbl_wc_arcsinh"] = nbl_sdata.update_annotated_regions_metadata(table=nbl_adata)

In [ ]:
fov = "NBL-1-R5C8"

nbl_sdata.filter_by_coordinate_system(fov).pl.render_labels(
    element=f"{fov}_whole_cell",
    color="spatialleiden",
    outline_alpha=0.99,
    scale="full",
    contour_px=None,
    # size=1,
    fill_alpha=1,
).pl.show(dpi=300)